# From-Scratch MLP (Notebook)

This notebook runs the same training workflow in `models/mlp_from_scratch.py` so generated artifacts and reported metrics stay consistent.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "models" / "mlp_from_scratch.py").exists() and (candidate / "dataset").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root from current working directory.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
print(f"Working directory: {REPO_ROOT}")

TRAIN_CSV = "dataset/NBA_Train.csv"
VALIDATION_CSV = "dataset/NBA_Validation.csv"
TEST_CSV = "dataset/NBA_Test.csv"
OUTPUT_DIR = "outputs/mlp/repro_run"

# Defaults below match the reported full run in outputs/mlp/mid_checkpoint/mlp_results.json.
TEAM_TOP_K = 0
EPOCHS = 220
PATIENCE = 40
BATCH_SIZE = 512
SEED = 5368
FEATURE_KEEP_OPTIONS = "0,320"

# Set True for a quick smoke run.
QUICK = False


Working directory: /Users/arca/Documents/Cornell™/Classes™/INFO5368/PAML-Final-Project


In [2]:
cmd = [
    sys.executable,
    "models/mlp_from_scratch.py",
    "--train",
    TRAIN_CSV,
    "--validation",
    VALIDATION_CSV,
    "--test",
    TEST_CSV,
    "--output-dir",
    OUTPUT_DIR,
    "--team-top-k",
    str(TEAM_TOP_K),
    "--epochs",
    str(EPOCHS),
    "--patience",
    str(PATIENCE),
    "--batch-size",
    str(BATCH_SIZE),
    "--seed",
    str(SEED),
    "--feature-keep-options",
    FEATURE_KEEP_OPTIONS,
]

if QUICK:
    cmd.append("--quick")

print("Running command:")
print(" ".join(cmd))
subprocess.run(cmd, check=True)


Running command:
/Users/arca/.pyenv/versions/3.14.2/bin/python3 models/mlp_from_scratch.py --train dataset/NBA_Train.csv --validation dataset/NBA_Validation.csv --test dataset/NBA_Test.csv --output-dir outputs/mlp/repro_run --team-top-k 0 --epochs 220 --patience 40 --batch-size 512 --seed 5368 --feature-keep-options 0,320


run 01/16 mode=single hidden=16 opt=sgd loss=cross_entropy features=463 val_macro_f1=0.6285 val_binary_f1=0.7389 val_binary_auc=0.9913
run 02/16 mode=single hidden=32 opt=adam loss=cross_entropy features=463 val_macro_f1=0.6289 val_binary_f1=0.7835 val_binary_auc=0.9928
run 03/16 mode=single hidden=64 opt=adam loss=cross_entropy features=463 val_macro_f1=0.6458 val_binary_f1=0.7895 val_binary_auc=0.9934
run 04/16 mode=single hidden=128 opt=adam loss=cross_entropy features=463 val_macro_f1=0.6393 val_binary_f1=0.7784 val_binary_auc=0.9938
run 05/16 mode=single hidden=64 opt=adam loss=focal features=463 val_macro_f1=0.6025 val_binary_f1=0.7892 val_binary_auc=0.9938
run 06/16 mode=single hidden=128 opt=adam loss=focal features=463 val_macro_f1=0.6234 val_binary_f1=0.7486 val_binary_auc=0.9926
run 07/16 mode=two_stage hidden=64 opt=adam loss=cross_entropy features=463 val_macro_f1=0.6347 val_binary_f1=0.7843 val_binary_auc=0.9934
run 08/16 mode=two_stage hidden=128 opt=adam loss=focal feat

CompletedProcess(args=['/Users/arca/.pyenv/versions/3.14.2/bin/python3', 'models/mlp_from_scratch.py', '--train', 'dataset/NBA_Train.csv', '--validation', 'dataset/NBA_Validation.csv', '--test', 'dataset/NBA_Test.csv', '--output-dir', 'outputs/mlp/repro_run', '--team-top-k', '0', '--epochs', '220', '--patience', '40', '--batch-size', '512', '--seed', '5368', '--feature-keep-options', '0,320'], returncode=0)

In [3]:
results_path = Path(OUTPUT_DIR) / "mlp_results.json"
if not results_path.exists():
    raise FileNotFoundError(f"Missing expected file: {results_path}")

results = json.loads(results_path.read_text(encoding="utf-8"))

summary = {
    "best_hyperparameters": results["best_run"]["hyperparameters"],
    "validation_macro_f1": results["validation_metrics"]["macro_f1"],
    "validation_binary_f1": results["validation_metrics"]["binary_drafted"]["f1"],
    "validation_binary_auc": results["validation_metrics"]["binary_drafted"]["auc"],
    "test_macro_f1": results["test_metrics"]["macro_f1"],
    "test_binary_f1": results["test_metrics"]["binary_drafted"]["f1"],
    "test_binary_auc": results["test_metrics"]["binary_drafted"]["auc"],
    "binary_threshold": results["binary_threshold"]["threshold"],
}

print(json.dumps(summary, indent=2))

test_pred_path = Path(OUTPUT_DIR) / "mlp_predictions_test.csv"
if test_pred_path.exists():
    display(pd.read_csv(test_pred_path).head())


{
  "best_hyperparameters": {
    "hidden_dim": 64,
    "learning_rate": 0.001,
    "l2": 0.001,
    "activation": "relu",
    "class_weight_mode": "sqrt_balanced",
    "batch_size": 512,
    "max_epochs": 220,
    "patience": 40,
    "seed": 5371,
    "optimizer": "adam",
    "loss": "cross_entropy",
    "focal_gamma": 0.0,
    "mode": "single"
  },
  "validation_macro_f1": 0.6457758611073978,
  "validation_binary_f1": 0.7894736842105263,
  "validation_binary_auc": 0.993446270693645,
  "test_macro_f1": 0.5227676029649745,
  "test_binary_f1": 0.47244094488188976,
  "test_binary_auc": 0.9559192825112107,
  "binary_threshold": 0.3667554063579299
}


,player_name,year,true_draft_status,pred_draft_status,pred_label,prob_undrafted,prob_1st_round,prob_2nd_round,prob_drafted_any,prob_drafted_any_raw,pred_drafted_any_thresholded
0,Troy Holston,2021,0,0,Undrafted,0.999927,0.000030,0.000043,0.000010,0.000073,0
1,Isaiah Felder,2021,0,0,Undrafted,0.999984,0.000014,0.000002,0.000002,0.000016,0
2,Thomas Bruce,2021,0,0,Undrafted,0.997412,0.000430,0.002158,0.000351,0.002588,0
3,Tyler Underwood,2021,0,0,Undrafted,0.999977,0.000019,0.000005,0.000003,0.000023,0
4,Jalen Coleman-Lands,2021,0,0,Undrafted,0.996494,0.000061,0.003445,0.000476,0.003506,0


In [4]:
reference_path = Path("outputs/mlp/mid_checkpoint/mlp_results.json")
if reference_path.exists():
    reference = json.loads(reference_path.read_text(encoding="utf-8"))
    comparison = {
        "validation_macro_f1_delta": summary["validation_macro_f1"] - reference["validation_metrics"]["macro_f1"],
        "validation_binary_f1_delta": summary["validation_binary_f1"] - reference["validation_metrics"]["binary_drafted"]["f1"],
        "validation_binary_auc_delta": summary["validation_binary_auc"] - reference["validation_metrics"]["binary_drafted"]["auc"],
        "test_macro_f1_delta": summary["test_macro_f1"] - reference["test_metrics"]["macro_f1"],
        "test_binary_f1_delta": summary["test_binary_f1"] - reference["test_metrics"]["binary_drafted"]["f1"],
        "test_binary_auc_delta": summary["test_binary_auc"] - reference["test_metrics"]["binary_drafted"]["auc"],
        "binary_threshold_delta": summary["binary_threshold"] - reference["binary_threshold"]["threshold"],
    }
    print(json.dumps(comparison, indent=2))
else:
    print("Reference file not found at outputs/mlp/mid_checkpoint/mlp_results.json")


{
  "validation_macro_f1_delta": -0.009887722533151622,
  "validation_binary_f1_delta": 0.0019089173711480667,
  "validation_binary_auc_delta": 5.636978579470764e-05,
  "test_macro_f1_delta": -0.004458475783583382,
  "test_binary_f1_delta": 0.0018527095877720612,
  "test_binary_auc_delta": 0.0008642478597634673,
  "binary_threshold_delta": 0.03289924760716928
}
